In [30]:
import pandas as pd, plotly.express as pe, duckdb as db

In [2]:
from pathlib import Path

csv_path = Path("../data/ato_sessions.csv")
df = pd.read_csv(csv_path)
df.head()

,session_id,timestamp,user_id,kyc_tier,kyc_tier_enc,account_age_days,is_new_device,is_new_ip_country,usual_country,days_since_last_login,session_to_withdrawal_secs,is_new_withdrawal_address,amount_btc,amount_vs_user_avg,hour_of_day,hour_deviation,is_fraud
0,S00051306,2025-01-01 00:00:00,U0001208,basic,1,1522,0,0,DE,7.15,1357,0,0.086541,0.9646,20,1.1,0
1,S00079169,2025-01-01 00:00:00,U0001615,basic,1,556,0,0,BR,0.77,1270,0,0.605012,0.6640,9,1.1,0
2,S00044928,2025-01-01 00:00:00,U0003138,full,2,1376,0,0,US,2.95,825,0,5.450799,1.5654,11,2.0,0
3,S00028335,2025-01-01 00:01:00,U0001874,full,2,1154,0,0,DE,11.27,886,0,2.929582,0.8675,6,0.7,0
4,S00052082,2025-01-01 00:04:00,U0005709,basic,1,1703,0,0,CA,0.11,1837,0,0.519773,0.8760,21,0.8,0


In [ ]:
import pandas as pd

df = pd.read_csv("../data/ato_sessions.csv")
df["is_fraud"] = df["is_fraud"].astype("category")
#print(df.shape)
#print(df["is_fraud"].value_counts(normalize=True))

session_id                    0
timestamp                     0
user_id                       0
kyc_tier                      0
kyc_tier_enc                  0
account_age_days              0
is_new_device                 0
is_new_ip_country             0
usual_country                 0
days_since_last_login         0
session_to_withdrawal_secs    0
is_new_withdrawal_address     0
amount_btc                    0
amount_vs_user_avg            0
hour_of_day                   0
hour_deviation                0
is_fraud                      0
dtype: int64
is_fraud
0    0.94
1    0.06
Name: proportion, dtype: float64


In [ ]:
#Correlation analysis, matrix 
#Select numeric columns for correlation
feature_cols = ['kyc_tier_enc', 'account_age_days', 'is_new_device', 'is_new_ip_country', 
                'days_since_last_login', 'session_to_withdrawal_secs', 'is_new_withdrawal_address', 
                'amount_btc', 'amount_vs_user_avg', 'hour_of_day', 'hour_deviation', 'is_fraud']

corr_matrix = df[feature_cols].corr()

# Display correlation with is_fraud
print("Correlations with is_fraud:")
print(corr_matrix['is_fraud'].sort_values(ascending=False))

pe.imshow(corr_matrix, text_auto=True, aspect="auto", width=1000, height=800).show()

Correlations with is_fraud:
is_fraud                      1.000000
amount_vs_user_avg            0.853950
days_since_last_login         0.827641
hour_deviation                0.678639
is_new_device                 0.677907
is_new_ip_country             0.656810
is_new_withdrawal_address     0.561681
amount_btc                    0.388883
account_age_days              0.005524
kyc_tier_enc                 -0.000776
hour_of_day                  -0.039695
session_to_withdrawal_secs   -0.135882
Name: is_fraud, dtype: float64


In [23]:

df.groupby("is_fraud")[[
    "is_new_device", "is_new_ip_country", "days_since_last_login",
    "session_to_withdrawal_secs", "is_new_withdrawal_address",
    "amount_vs_user_avg", "hour_deviation"
]].median().T

is_fraud,0,1
is_new_device,0.0000,1.00000
is_new_ip_country,0.0000,1.00000
days_since_last_login,1.6000,66.15500
session_to_withdrawal_secs,1807.0000,55.00000
is_new_withdrawal_address,0.0000,1.00000
amount_vs_user_avg,1.0021,5.98235
hour_deviation,1.6000,10.00000


plot relationship between days without logging in and fraud 

In [18]:
summary = (
    df.groupby("is_fraud", as_index=False)["days_since_last_login"]
      .median()
      .rename(columns={"days_since_last_login": "median_days_since_last_login"})
)

pe.bar(summary, x="is_fraud", y="median_days_since_last_login").show()



In [28]:
display(df.groupby("is_fraud").indices)

{0: array([    0,     1,     2, ..., 79997, 79998, 79999], shape=(75200,)),
 1: array([    7,    44,    86, ..., 79969, 79970, 79971], shape=(4800,))}

In [36]:
ddf = db.sql("select is_fraud, median(session_to_withdrawal_secs) as session_to_withdrawal_secs from df group by is_fraud").df()
ddf['is_fraud'] = ddf["is_fraud"].astype(str)
pe.bar(ddf, x="is_fraud", y="session_to_withdrawal_secs").show()

In [25]:
summary = (
    df.groupby("is_fraud", as_index=False)['session_to_withdrawal_secs'].median().rename(columns={'sum of session_to_withdrawal_secs':'session_to_withdrawal_secs'})
)
pe.histogram(df, x='session_to_withdrawal_secs', color='is_fraud').show()

In [39]:
db.sql("select is_fraud, count(kyc_tier) from df group by is_fraud")

┌──────────┬─────────────────┐
│ is_fraud │ count(kyc_tier) │
│  int64   │      int64      │
├──────────┼─────────────────┤
│        0 │           75200 │
│        1 │            4800 │
└──────────┴─────────────────┘